In [26]:
import pandas as pd
import requests
import time
import os
import geopandas as gpd

In [27]:
# File paths
DATA_DIR   = '../data/raw/'
OUTPUT_DIR = '../data/processed/'

# Create the processed folder if it doesn't exist yet
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load the AIC facilities file — postal_code must stay as text (dtype=str)
aic = pd.read_csv(DATA_DIR + 'DataAICDemFacilitiesSnapshot20260312.csv',
                  dtype={'postal_code': str})

print(f'Loaded {len(aic)} facilities')
print(aic.head())

df = aic.copy()

Loaded 148 facilities
   facility_id                                      facility_name  \
0            1               4S Eldercare Centre @ Eunos Crescent   
1            2                      AWWA Dementia Day Care Centre   
2            3             AWWA Dementia Day Care Centre @ Yishun   
3            4                    Active Global @ Fernvale Glades   
4            5  Active Global Ghim Moh Active Ageing Centre (C...   

                                        full_address postal_code snapshot_date  
0         31A Eunos Crescent #06-01 Singapore 401031      401031     3/12/2026  
1  123 Ang Mo Kio Avenue 6 #01-4035 Singapore 560123      560123     3/12/2026  
2       740 Yishun Avenue 5 #01-490 Singapore 760740      760740     3/12/2026  
3      460 Sengkang West Way #01-02 Singapore 790460      790460     3/12/2026  
4          31A Ghim Moh Link #01-11 Singapore 272031      272031     3/12/2026  


## Cell 3 — Geocoding Function (Revised Scope)

Original plan: query OneMap for both coordinates and planning area name per facility.

Revised: OneMap's search endpoint does not return planning area directly. 
A separate planning area API endpoint exists but would require a second API call 
per facility (296 total calls), increasing failure risk and API dependency.

Decision: use OneMap to retrieve latitude and longitude only. Planning area 
assignment will be handled via spatial join against the URA Master Plan 2019 
GeoJSON in 03_merge_and_compute.py. This keeps the boundary data on a single 
consistent snapshot and reduces external dependencies.

    def get_planning_area(postal_code):
        """
        Takes a postal code (string), queries the OneMap API,
        and returns the planning area name as a string.
           Returns None if the query fails or returns no results.
        """
    
    # Step 1: Build the URL
    base_url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": postal_code,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    
    # Step 2: Make the API request inside a try/except block
    try:
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        # Step 3: Check if results exist
        if data["found"] > 0:
            result = data["results"][0]
            planning_area = result.get("SEARCHVAL", None)
            return planning_area
        else:
            print(f"No results found for postal code: {postal_code}")
            return None
    
    except requests.exceptions.RequestException as e:
        print(f"Request failed for postal code {postal_code}: {e}")
        return None

    #Test on one postal code
    test_result = get_planning_area("560123")
    print(test_result)

In [28]:
def get_coordinates(postal_code):
    base_url = "https://www.onemap.gov.sg/api/common/elastic/search"
    params = {
        "searchVal": postal_code,
        "returnGeom": "Y",
        "getAddrDetails": "Y",
        "pageNum": 1
    }
    
    try:
        response = requests.get(base_url, params=params, timeout=10)
        response.raise_for_status()
        data = response.json()
        
        if data["found"] > 0:
            result = data["results"][0]
            return {
                "latitude": result["LATITUDE"],
                "longitude": result["LONGITUDE"]
            }
        else:
            print(f"No results found for postal code: {postal_code}")
            return None
    
    except requests.exceptions.RequestException as e:
        print(f"Request failed for postal code {postal_code}: {e}")
        return None

# Test on one postal code
test_result = get_coordinates("560123")
print(test_result)

{'latitude': '1.37048118793194', 'longitude': '103.844805800791'}


## Data Quality Note

Facility ID 1920 (Kwong Wai Shiu Care @ McNair, 113 McNair Road) had an 
incorrect postal code in the source CSV (328972). Correct postal code 321113 
verified against the facility's official website. Source CSV corrected before 
re-running pipeline.

In [29]:
import os

GEOCODED_PATH = OUTPUT_DIR + 'facilities_geocoded.csv'

if os.path.exists(GEOCODED_PATH):
    df = pd.read_csv(GEOCODED_PATH, dtype={'postal_code': str})
    print(f"Loaded existing geocoded file with {len(df)} facilities")

else:
    df = aic.copy()
    
    for index, row in df.iterrows():
        postal_code = row["postal_code"]
        result = get_coordinates(postal_code)
        
        if result is not None:
            df.at[index, "latitude"] = float(result["latitude"])
            df.at[index, "longitude"] = float(result["longitude"])
        else:
            df.at[index, "latitude"] = None
            df.at[index, "longitude"] = None
            print(f"FAILED: {postal_code}")
        
        time.sleep(0.5)
    
    df.to_csv(GEOCODED_PATH, index=False)
    print("Done — results saved to file")
    print(f"Failed: {df['latitude'].isna().sum()} facilities")

Loaded existing geocoded file with 148 facilities


In [30]:
gdf = gpd.read_file(DATA_DIR + 'DataMasterPlan2019PlngAreaBoundaryNoSea.geojson')
print(gdf.shape)
print(gdf.columns.tolist())
print(gdf.head())

(55, 11)
['OBJECTID', 'PLN_AREA_N', 'PLN_AREA_C', 'CA_IND', 'REGION_N', 'REGION_C', 'INC_CRC', 'FMEL_UPD_D', 'SHAPE.AREA', 'SHAPE.LEN', 'geometry']
   OBJECTID     PLN_AREA_N PLN_AREA_C CA_IND        REGION_N REGION_C  \
0         1          BEDOK         BD      N     EAST REGION       ER   
1         2       BOON LAY         BL      N     WEST REGION       WR   
2         3    BUKIT BATOK         BK      N     WEST REGION       WR   
3         4    BUKIT MERAH         BM      N  CENTRAL REGION       CR   
4         5  BUKIT PANJANG         BP      N     WEST REGION       WR   

            INC_CRC      FMEL_UPD_D    SHAPE.AREA     SHAPE.LEN  \
0  5F00E6FF084F3364  20191223152014  2.173397e+07  21864.234336   
1  C96AED188C00B2FC  20191223152014  8.282808e+06  18542.688792   
2  3BEC4C829160F28A  20191223152014  1.114018e+07  15236.442232   
3  4850795BB0B6A4F7  20191223152014  1.446120e+07  29161.409834   
4  656F87D23D6DAB02  20191223152014  9.019940e+06  15891.853279   

          

In [34]:
from shapely.geometry import Point

aic_gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["longitude"], df["latitude"]),
    crs="EPSG:4326"
)

print(aic_gdf.shape)
print(aic_gdf[["facility_name", "latitude", "longitude", "geometry"]].head())

(148, 8)
                                       facility_name  latitude   longitude  \
0               4S Eldercare Centre @ Eunos Crescent  1.320034  103.901616   
1                      AWWA Dementia Day Care Centre  1.370481  103.844806   
2             AWWA Dementia Day Care Centre @ Yishun  1.430124  103.832153   
3                    Active Global @ Fernvale Glades  1.394261  103.869619   
4  Active Global Ghim Moh Active Ageing Centre (C...  1.309329  103.784354   

                    geometry  
0  POINT (103.90162 1.32003)  
1  POINT (103.84481 1.37048)  
2  POINT (103.83215 1.43012)  
3  POINT (103.86962 1.39426)  
4  POINT (103.78435 1.30933)  


In [36]:
aic_joined = gpd.sjoin(aic_gdf, gdf[["PLN_AREA_N", "geometry"]], how="left", predicate="within")

print(aic_joined.shape)
print(aic_joined[["facility_name", "postal_code", "latitude", "longitude", "PLN_AREA_N"]].head(10))

(148, 10)
                                       facility_name postal_code  latitude  \
0               4S Eldercare Centre @ Eunos Crescent      401031  1.320034   
1                      AWWA Dementia Day Care Centre      560123  1.370481   
2             AWWA Dementia Day Care Centre @ Yishun      760740  1.430124   
3                    Active Global @ Fernvale Glades      790460  1.394261   
4  Active Global Ghim Moh Active Ageing Centre (C...      272031  1.309329   
5  Active Global Telok Blangah Active Ageing Cent...      100092  1.276215   
6                         All Saints Home (Tampines)      529123  1.361206   
7  All Saints Silver Lifestyle Club @ Yishun Central      768898  1.430359   
8  All Saints Silver Lifestyle Club @ Yishun Fern...      760674  1.420139   
9     Care Corner Active Ageing Hub @ Toa Payoh East      311261  1.333726   

    longitude   PLN_AREA_N  
0  103.901616      GEYLANG  
1  103.844806   ANG MO KIO  
2  103.832153       YISHUN  
3  103.869619  

In [37]:
print(aic_joined["PLN_AREA_N"].isna().sum())
print(aic_joined["PLN_AREA_N"].nunique())

0
27


In [38]:
aic_joined[["facility_id", "facility_name", "full_address", "postal_code", "latitude", "longitude", "PLN_AREA_N"]].to_csv(
    OUTPUT_DIR + "facilities_with_planning_area.csv",
    index=False
)

print("Saved successfully")

Saved successfully
